In [ ]:
import torch
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    TPU_AVAILABLE = True
except ImportError:
    TPU_AVAILABLE = False
    
if TPU_AVAILABLE:
    DEVICE = torch_xla.device()
    !pip install soundfile -q
    !pip install torchinfo -q
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"USING DEVICE {DEVICE}")

In [ ]:
import os
import sys
import random
import math
import ast
import warnings
from pathlib import Path

# Audio processing modules
import soundfile as sf

# Data handling and visualization modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Skikit-learn preprocessing modules
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold

# PyTorch modules
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
import torchvision.transforms as TV
from torchinfo import summary
#/kaggle/input/competitions/birdclef-2026/

In [ ]:
EXAMPLE_AUDIO_PATH = "/kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1114648.ogg"

SAMPLE_RATE = 32000

#transform params
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 320
SAMPLE_DURATION = 5

N_SAMPLES = SAMPLE_RATE * SAMPLE_DURATION   # Number of samples of preprocessed audio files
F_MIN       = 20                            # Minimum frequency
F_MAX       = 16000                         # Maximum frequency
BATCH_SIZE = 64

WARM_START = True
SUBMITTING = True

In [ ]:
def generate_audio_slices(audio_full_paths):
    slices =[]
    i = 0
    for audio_full_path in tqdm(audio_full_paths):
        # if i == 10:
        #     break
        # i += 1
        info = sf.info(audio_full_path)
        total_samples = info.frames
    
        start_frame = 0
        end_frame = N_SAMPLES
    
        while end_frame < total_samples: #make sure the next frame doesnt go over
            #replace .ogg with required submission id
            row_id = os.path.basename(audio_full_path).replace(".ogg", f"_{int(end_frame/SAMPLE_RATE)}") 
           
            slices.append((row_id, audio_full_path, start_frame, end_frame))
            start_frame += N_SAMPLES
            end_frame += N_SAMPLES

    return pd.DataFrame(slices, columns=["row_id", "full_path", "start_frame", "end_frame"])

In [ ]:
#trainval = pd.read_csv("/kaggle/working/processed_trainval.csv")
if os.path.exists("/kaggle/working/processed_trainval.csv"):
    trainval = pd.read_csv("/kaggle/working/processed_trainval.csv")
else:
    trainval = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
    #print(trainval.head()0
    #conv trainval start/end sec -> frames
    trainval_extra = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
    trainval_extra["extra_start_frame"] = pd.to_timedelta(trainval_extra['start']).dt.total_seconds().astype(int) * SAMPLE_RATE
    trainval_extra["extra_end_frame"] = pd.to_timedelta(trainval_extra['end']).dt.total_seconds().astype(int) * SAMPLE_RATE
    trainval_extra.drop(columns=["start","end"], inplace=True)
    #print(trainval_extra.head())
    taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
    
    # Merge and standardize label informations from different columns of data source tables (all labels in a single string seperated by ;)
    trainval['all_labels'] = trainval.apply(lambda row: ';'.join([row['primary_label']] + ast.literal_eval(row['secondary_labels'])), axis=1)
    trainval_extra = trainval_extra.drop_duplicates().reset_index(drop=True).rename(columns={'primary_label': 'all_labels'})
    trainval_extra['primary_label'] = trainval_extra['all_labels'].str.split(';').str[0]
    trainval_extra['secondary_labels'] = trainval_extra['all_labels'].str.split(';').str[1:]
    
    # Include full paths in the data source tables
    trainval['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_audio/' + trainval['filename']
    trainval_extra['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_soundscapes/' + trainval_extra['filename']
    
    # Create a list of 5 sec slices of audio files and merge information into trainval
    audio_full_paths = trainval['full_path'].unique().tolist()
    slices_df = generate_audio_slices(audio_full_paths)
    trainval = pd.merge(trainval, slices_df, on='full_path', how='right')
    
    # Merge the two datasets into one single dataframe
    trainval = pd.concat([trainval, trainval_extra], axis=0)
    trainval["start_frame"] = trainval["start_frame"].combine_first(trainval["extra_start_frame"])
    trainval["end_frame"] = trainval["end_frame"].combine_first(trainval["extra_end_frame"])
    trainval.drop(columns=["extra_start_frame","extra_end_frame"], inplace=True)
    
    trainval.to_csv("processed_trainval.csv")

print(trainval.head())

In [ ]:
#one hot encoder
all_unique_labels = trainval['all_labels'].str.split(';').explode().unique()
mlb = MultiLabelBinarizer()
mlb.fit([all_unique_labels])
print(f"\nTotal unique labels: {len(all_unique_labels)}")
print(f"Label classes: {mlb.classes_}")

In [ ]:
class AnimalSoundDataset(Dataset):
    def __init__(self, dataframe, mlb=mlb, submitting=False):
        self.dataframe = dataframe
        self.mlb = mlb
        self.sample_rate = SAMPLE_RATE
        self.max_audio_samples = N_SAMPLES
        self.submitting = submitting
        
    def __len__(self):
        return len(self.dataframe)

    def _load_audio_slice_at(self, path, start_sample, end_sample):
        stereo_waveform, sr = sf.read(path, start=start_sample.astype(int), stop=end_sample.astype(int), dtype='float32', always_2d=True)
        waveform = stereo_waveform.mean(axis=1)
        waveform = torch.from_numpy(waveform).to(torch.float32).contiguous()
        waveform = waveform.unsqueeze(0)
        return waveform, sr

    def _process_audio(self, audio_path_full, start_samples, end_samples):
        # Cut 5 sec audio sample with defined starting time from given audio file
        waveform, sr = self._load_audio_slice_at(audio_path_full, start_samples, end_samples)

        # Resample waveform if necessary
        if sr != self.sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Padding or truncating sample if necessary
        current_n_samples = waveform.shape[1]
        if current_n_samples < self.max_audio_samples:
            padding_needed = self.max_audio_samples - current_n_samples
            waveform = F.pad(waveform, (0, padding_needed))
        elif current_n_samples > self.max_audio_samples:
            waveform = waveform[:, :self.max_audio_samples]
            
        return waveform

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        row_id = row['row_id']
        audio_path_full = row['full_path']
        start_samples = row['start_frame']
        end_samples = row['end_frame']
        waveform = self._process_audio(audio_path_full, start_samples, end_samples)
        if not self.submitting:
            labels = row['all_labels'].split(';')
            one_hot_labels = torch.tensor(self.mlb.transform([labels]).squeeze(0), dtype=torch.float32)
            return (waveform, one_hot_labels)
        else:
            return (waveform, row_id)

In [ ]:
class MelSpecHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,        # FFT window size
            hop_length=HOP_LENGTH,    # step size between frames
            n_mels=N_MELS,        # number of mel bands
            f_min=F_MIN,
            f_max=F_MAX,
            power=2.0          # power spectrogram (2.0 = energy)
        )
    def forward(self, waveform):
        self.amp_to_db_transform = T.AmplitudeToDB(top_db=80)
        mel_S_db = self.amp_to_db_transform(self.mel_transform(waveform))
        smp_min = mel_S_db.amin(dim=(1,2), keepdim=True)
        smp_max = mel_S_db.amax(dim=(1,2), keepdim=True)
        mel_S_db_norm = (mel_S_db - smp_min) / (smp_max - smp_min + 1e-8)
        return mel_S_db_norm

# class ResNetHead(nn.Module):
#     def __init__(self):
#         super().__init__()
#         resnet = models.resnet18(weights="DEFAULT")
#         resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False) #adapt to spectrogram
#         resnet_modules = list(resnet.children())[:-1] #output fc
#         self.resnet = nn.Sequential(*resnet_modules)

#     def forward(self, x):
#         return self.resnet(x)

class EfficientNetHead(nn.Module):
    def __init__(self):
        super().__init__()
        effnet = models.efficientnet_b1(weights=None)
        effnet_modules = list(effnet.children())[:-1] #output fc
        self.effnet = nn.Sequential(*effnet_modules)
        self.bottleneck = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1280,512), #contract to fit classifier
            nn.GELU(),
        )

    def forward(self, x):
        # x = torch.concat([x,x,x], dim=1) #make it 3channel
        x = x.expand(-1,3,-1,-1)
        x = self.effnet(x)
        x = self.bottleneck(x)
        return x

class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.msh = MelSpecHead()
        self.fe = EfficientNetHead()
        self.classifier = nn.Sequential(
            nn.Linear(512,512),
            nn.GELU(),
            nn.Linear(512,num_classes)
        )
        
    def forward(self,x):
        x = self.msh(x)
        x = self.fe(x)
        x = self.classifier(x)
        return x

In [ ]:
summary(Classifier(234))

In [ ]:
class MultiLabelLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, weight=None, pos_weight=None, reduction='mean') #sigmoid already done internally
        return bce_loss

In [ ]:
if not SUBMITTING:
    NUM_CLASSES = len(mlb.classes_)
    LEARNING_RATE = 1e-5
    NUM_EPOCHS = 15
    NUM_WORKERS = 4
    
    def get_metrics(logits_np, targets_np):
        present_classes_mask = np.any(targets_np == 1, axis=0) #columns
        filtered_logits = logits_np[:, present_classes_mask]
        filtered_targets = targets_np[:, present_classes_mask]
        probs = 1.0 / (1.0 + np.exp(-filtered_logits))
        map_score = average_precision_score(filtered_targets, probs, average="macro")
        roc_score = roc_auc_score(filtered_targets, probs, average="macro")
        return map_score, roc_score
    
    def train_one_epoch(model, loader, criterion, optimizer, device):
        model.train()
        running = 0.0
        n = 0
        for waveforms, targets in tqdm(loader, desc="train", leave=False):
            waveforms = waveforms.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(waveforms)
            loss = criterion(logits, targets)
            loss.backward()
            if TPU_AVAILABLE:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()
            bs = waveforms.size(0)
            running += loss.item() * bs
            n += bs
        return running / max(n, 1)
    
    @torch.no_grad()
    def evaluate(model, loader, criterion, device):
        model.eval()
        running = 0.0
        n = 0
        all_logits = []
        all_targets = []
        
        for waveforms, targets in tqdm(loader, desc="val", leave=False):
            waveforms = waveforms.to(device)
            targets = targets.to(device)
            logits = model(waveforms)
            loss = criterion(logits, targets)
            bs = waveforms.size(0)
            running += loss.item() * bs
            n += bs
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
        logits_np = np.concatenate(all_logits, axis=0)
        targets_np = np.concatenate(all_targets, axis=0)
    
        map_score, roc_score = get_metrics(logits_np, targets_np)
        return running / max(n, 1), map_score, roc_score
    
    train_df = trainval.sample(frac=0.9, random_state=42)
    val_df = trainval.drop(train_df.index)
    
    train_loader = DataLoader(
        AnimalSoundDataset(train_df.reset_index(drop=True), submitting=False),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE == "cuda",
        prefetch_factor=2
    )
    val_loader = DataLoader(
        AnimalSoundDataset(val_df.reset_index(drop=True), submitting=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE == "cuda",
        prefetch_factor=2
    )
    
    model = Classifier(NUM_CLASSES).to(DEVICE)
    criterion = MultiLabelLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    if WARM_START:
        try:
            checkpoint = torch.load("/kaggle/working/checkpoint.pth", map_location=DEVICE)
            model.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            print("WARM START SUCCESSFUL")
        except Exception as e:
            print(f"WARM START EXCEPTION: {e}")
    
    train_losses = []
    val_losses = []
    val_map_scores = []
    val_roc_scores = []
    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_map, val_roc = evaluate(model, val_loader, criterion, DEVICE)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_map_scores.append(val_map)
        val_roc_scores.append(val_roc)
        print(f"epoch {epoch + 1}/{NUM_EPOCHS} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_mAP={val_map:.4f} val_roc={val_roc:.4f}")
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }, "checkpoint.pth")
    
    fig, axes = plt.subplots(1,2)
    axes[0].plot(train_losses, marker='o')
    axes[0].plot(val_losses, marker='o')
    axes[1].plot(val_map_scores, marker='o')
    axes[1].plot(val_roc_scores, marker='o')
    plt.show()

In [ ]:
if SUBMITTING:
    model = Classifier(234)
    checkpoint = torch.load("/kaggle/input/models/fluidize/birdclef-v3/pytorch/default/1/checkpoint.pth", map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])
    model.to(DEVICE)
    
    test_path = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
    #test_path = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
    with os.scandir(test_path) as audio_files:
        test_audio_full_paths = [os.path.abspath(file.path) for file in audio_files]
        #test_audio_full_paths.remove("/kaggle/input/competitions/birdclef-2026/test_soundscapes/readme.txt")
    testset = generate_audio_slices(test_audio_full_paths)
    test_loader = DataLoader(
        AnimalSoundDataset(testset.reset_index(drop=True), submitting=True),
        batch_size=1,
        shuffle=False,
        num_workers=4,
        pin_memory=DEVICE == "cuda",
        prefetch_factor=2
    )
    
    test_results = []
    
    with torch.no_grad():
        model.eval()
        for waveform, row_id in tqdm(test_loader):
            waveform = waveform.to(DEVICE)
            logits = F.sigmoid(torch.squeeze(model(waveform)))
            logits = logits.detach().cpu().numpy()
            
            test_row = [row_id[0], *logits] #undo dataloader batchdim for strings
            test_results.append(test_row)

        out_columns = ['row_id'] + list(mlb.classes_)
            
    OUTPUT = pd.DataFrame(test_results, columns=out_columns)
    # print(OUTPUT)
    # print(out_columns)
    OUTPUT.to_csv("submission.csv", index=False)
            
        